In [4]:
import sys
sys.path.insert(0, '..')

In [5]:
from premia_junior.models.black_scholes import BlackScholes
from premia_junior.methods.analytical import Analytical
from premia_junior.products.european import EuropeanCall
from premia_junior.engine.pricer import Pricer

model = BlackScholes(mu=0.05, sigma=0.2)
method = Analytical()
product = EuropeanCall(K=100)

pricer = Pricer(model=model, method=method, product=product)
result = pricer.price(S0=100, T=1.0, r=0.05)
print(result)

PricingResult(price=10.450583572185565, confidence_interval=None, metadata={'method': 'Analytical', 'model': 'BlackScholes', 'd1': 0.35000000000000003, 'd2': 0.15000000000000002})


In [6]:
from premia_junior.methods.monte_carlo import MonteCarlo

mc = MonteCarlo(n_paths=100000, n_steps=252, seed=42)
pricer_mc = Pricer(model=model, method=mc, product=product)
result_mc = pricer_mc.price(S0=100, T=1.0, r=0.05)
print(result_mc)

PricingResult(price=10.482858080322606, confidence_interval=(10.390945307384213, 10.574770853260997), metadata={'Method': 'MonteCarlo', 'Model': 'BlackScholes', 'Product': 'EuropeanCall'})


In [7]:
import numpy as np
from premia_junior.models.black_scholes import BlackScholes
from premia_junior.methods.CRR import CRR
from premia_junior.methods.analytical import Analytical
from premia_junior.products.european import EuropeanPut
from premia_junior.products.american import AmericanPut
from premia_junior.engine.pricer import Pricer

model = BlackScholes(mu=0.05, sigma=0.2)
S0, K, T, r = 100, 100, 1.0, 0.05

# European put — CRR vs Analytical
eu_put = EuropeanPut(K=K)
crr = CRR(n_steps=500)

r1 = Pricer(model=model, method=Analytical(), product=eu_put).price(S0, T, r)
r2 = Pricer(model=model, method=crr, product=eu_put).price(S0, T, r)
print(f"European put — Analytical: {r1.price:.4f}, CRR: {r2.price:.4f}")

# American put — CRR
am_put = AmericanPut(K=K)
r3 = Pricer(model=model, method=crr, product=am_put).price(S0, T, r)
print(f"American put — CRR: {r3.price:.4f}")

European put — Analytical: 5.5735, CRR: 5.5695
American put — CRR: 6.0888


In [8]:
from premia_junior.models.heston import Heston
from premia_junior.methods.monte_carlo import MonteCarlo
from premia_junior.products.european import EuropeanCall
from premia_junior.engine.pricer import Pricer

model = Heston(v0=0.04, kappa=1.5, theta=0.04, xi=0.3, rho=-0.7)
method = MonteCarlo(n_paths=100000, n_steps=252)
product = EuropeanCall(K=100)

result = Pricer(model=model, method=method, product=product).price(S0=100, T=1.0, r=0.05)
print(result)

Feller Condition is verified So the variance is always positive
PricingResult(price=10.371549533696681, confidence_interval=(10.297521023940755, 10.44557804345261), metadata={'Method': 'MonteCarlo', 'Model': 'Heston', 'Product': 'EuropeanCall'})


In [9]:
from premia_junior.models.black_scholes import BlackScholes
from premia_junior.methods.monte_carlo import MonteCarlo
from premia_junior.products.asian import DiscreteAsianCall
from premia_junior.engine.pricer import Pricer

model = BlackScholes(mu=0.05, sigma=0.2)
method = MonteCarlo(n_paths=100000, n_steps=252)
product = DiscreteAsianCall(K=100)

result = Pricer(model=model, method=method, product=product).price(S0=100, T=1.0, r=0.05)
print(result)


PricingResult(price=5.735492381014864, confidence_interval=(5.6861893904726175, 5.78479537155711), metadata={'Method': 'MonteCarlo', 'Model': 'BlackScholes', 'Product': 'DiscreteAsianCall'})


In [2]:
import sys
sys.path.insert(0, '..')

from premia_junior.models.black_scholes import BlackScholes
from premia_junior.methods.analytical import Analytical
from premia_junior.products.european import EuropeanCall
from premia_junior.engine.pricer import Pricer
from premia_junior.utils.greek import FiniteDifference
from premia_junior.utils.BaseGreekScheme import CentralDifference1

model   = BlackScholes(mu=0.05, sigma=0.2)
pricer  = Pricer(model=model, method=Analytical(), product=EuropeanCall(K=100))
fd      = FiniteDifference(CentralDifference1(), h=0.01)

# 1. with_params: the new model changes, the original doesn't
print(model.with_params(sigma=0.21).sigma, model.sigma)   # 0.21  0.2

# 2. the five Greeks
S0, T, r = 100, 1.0, 0.05
print("delta", fd.delta(pricer, S0, T, r))
print("gamma", fd.gamma(pricer, S0, T, r))
print("vega ", fd.vega(pricer, S0, T, r))
print("rho  ", fd.rho(pricer, S0, T, r))
print("theta", fd.theta(pricer, S0, T, r))

0.21 0.2
delta 0.6368306425763137
gamma 0.018762017077733617
vega  37.52098305590863
rho   53.224771767578005
theta -6.414077196225776


In [3]:
from premia_junior.methods.monte_carlo import MonteCarlo

S0, T, r = 100, 1.0, 0.05
product = EuropeanCall(K=100)

mc_crn   = MonteCarlo(n_paths=100000, n_steps=252, seed=42)    # same random numbers on every call
mc_nocrn = MonteCarlo(n_paths=100000, n_steps=252, seed=None)  # fresh random numbers on every call

pricer_crn   = Pricer(model=model, method=mc_crn,   product=product)
pricer_nocrn = Pricer(model=model, method=mc_nocrn, product=product)

for name, p in [("seed=42  ", pricer_crn), ("seed=None", pricer_nocrn)]:
    print(name, "delta", fd.delta(p, S0, T, r), " gamma", fd.gamma(p, S0, T, r))

seed=42   delta 0.6376292708700326  gamma 0.01713742324938039
seed=None delta 1.607813801286273  gamma -438.8996566666492
